In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.alerts import FlowDetector
from src.pipeline import DATA, RESULTS

det = FlowDetector(RESULTS, threshold=1.0)             # placeholder, replaced below
val = pd.read_parquet(DATA / "val.parquet").sample(200_000, random_state=11)
BUDGET = 0.05
det.threshold = float(np.quantile(det.score(val), 1 - BUDGET))
json.dump({"budget": BUDGET, "threshold": det.threshold, "model": "autoencoder-v2",
           "calibrated_on": "ToN-IoT benign validation split"},
          open(RESULTS / "alert_config.json", "w"), indent=2)
print("threshold:", det.threshold)


threshold: 0.005736944265663624


In [2]:
tune = pd.read_parquet(DATA / "tune.parquet").sample(50_000, random_state=11).sort_values("row_id")
n_alerts, hits = 0, 0
with open(RESULTS / "alerts_demo.jsonl", "w") as f:
    for i in range(0, len(tune), 5_000):                # small batches, like live traffic
        batch = tune.iloc[i:i + 5_000]
        truth = batch.set_index("row_id")["Attack"]
        for a in det.alerts(batch, id_prefix="tune"):
            a["_replay_truth"] = str(truth[a["flow_seq"]])   # only exists because the replay is labelled
            f.write(json.dumps(a) + "\n")
            n_alerts += 1
            hits += a["_replay_truth"] != "Benign"

n_benign = int((tune["Attack"] == "Benign").sum())
n_attack = len(tune) - n_benign
print(f"flows: {len(tune)} | alerts: {n_alerts}")
print(f"attack flows alerted: {hits}/{n_attack} ({hits / n_attack:.1%})")
print(f"benign flows alerted: {n_alerts - hits}/{n_benign} ({(n_alerts - hits) / n_benign:.1%})")
print(json.dumps(json.loads(open(RESULTS / "alerts_demo.jsonl").readline()), indent=2))

flows: 50000 | alerts: 22481
attack flows alerted: 20108/21588 (93.1%)
benign flows alerted: 2373/28412 (8.4%)
{
  "alert_id": "tune-7766125",
  "emitted_at": "2026-11-15T21:05:37+00:00",
  "flow_seq": 7766125,
  "src_ip": "192.168.1.32",
  "dst_ip": "192.168.1.1",
  "dst_port": 53,
  "protocol": 17,
  "score": 0.00852,
  "threshold": 0.00574,
  "severity": "low",
  "top_features": [
    "DNS_QUERY_TYPE",
    "DNS_QUERY_ID",
    "L7_PROTO"
  ],
  "model": "autoencoder-v2",
  "_replay_truth": "xss"
}
